In [3]:
# Import all required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor
from snowflake.snowpark.context import get_active_session


In [4]:
# Get Snowflake session
session = get_active_session()

# Load data from Snowflake
print("Loading data from Snowflake...")
df = session.sql("""
    SELECT * 
    FROM GROUP4_ASG2.FINAL_DATA.PROPWISE_MASTER
    
""").to_pandas()

print(f"✅ Loaded {len(df):,} rows and {len(df.columns)} columns")
print("\nFirst 5 rows:")
print(df.head())


   SALE_YEAR SALE_QUARTER  SALE_MONTH  POSTAL_CODE              ADDRESS  \
0       2020           Q3           8     543204.0  204C COMPASSVALE DR   
1       2020           Q3           8     543204.0  204C COMPASSVALE DR   
2       2020           Q3           8     543204.0  204C COMPASSVALE DR   
3       2020           Q3           8     543204.0  204C COMPASSVALE DR   
4       2020           Q3           8     543204.0  204C COMPASSVALE DR   

   RESALE_INDEX  RESALE_PRICE  MIN_SELLING_PRICE  MEDIAN_RESALE_PRICE  \
0         134.0        420000                NaN             560000.0   
1         134.0        420000                NaN                  NaN   
2         134.0        420000                NaN             410000.0   
3         134.0        420000                NaN             475000.0   
4         134.0        420000                NaN             590000.0   

   MAX_SELLING_PRICE  ... NEAREST_PARK_DISTANCE_M PARKS_WITHIN_1KM  \
0                NaN  ...               

,SALE_YEAR,SALE_QUARTER,SALE_MONTH,POSTAL_CODE,ADDRESS,RESALE_INDEX,RESALE_PRICE,MIN_SELLING_PRICE,MEDIAN_RESALE_PRICE,MAX_SELLING_PRICE,...,NEAREST_PARK_DISTANCE_M,PARKS_WITHIN_1KM,NEAREST_PRESCHOOL_M,PRESCHOOLS_WITHIN_1KM,NEAREST_CLINIC_M,CLINICS_WITHIN_500M,NEAREST_PHARMACY_M,NEAREST_HOSPITAL_M,HEALTHCARE_ACCESSIBILITY_SCORE,NEAREST_MALL_M
0,2020,Q3,8,543204.0,204C COMPASSVALE DR,134.0,420000,NaN,560000.0,NaN,...,1229.19,0.0,70.02,10.0,252.279761,8.0,503.796272,239.619364,100.0,495.136163
1,2020,Q3,8,543204.0,204C COMPASSVALE DR,134.0,420000,NaN,NaN,NaN,...,1229.19,0.0,70.02,10.0,252.279761,8.0,503.796272,239.619364,100.0,495.136163
2,2020,Q3,8,543204.0,204C COMPASSVALE DR,134.0,420000,NaN,410000.0,NaN,...,1229.19,0.0,70.02,10.0,252.279761,8.0,503.796272,239.619364,100.0,495.136163
3,2020,Q3,8,543204.0,204C COMPASSVALE DR,134.0,420000,NaN,475000.0,NaN,...,1229.19,0.0,70.02,10.0,252.279761,8.0,503.796272,239.619364,100.0,495.136163
4,2020,Q3,8,543204.0,204C COMPASSVALE DR,134.0,420000,NaN,590000.0,NaN,...,1229.19,0.0,70.02,10.0,252.279761,8.0,503.796272,239.619364,100.0,495.136163


In [5]:
# Basic data information
print("="*60)
print("DATA OVERVIEW")
print("="*60)
print(f"Shape: {df.shape}")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Check data types
print("\nData Types:")
print(df.dtypes.value_counts())

# Check missing values
print("\nMissing Values (top 10):")
missing = df.isnull().sum().sort_values(ascending=False)
print(missing.head(10))

# Check target variable
print("\nTarget Variable (RESALE_PRICE):")
print(f"  Mean: ${df['RESALE_PRICE'].mean():,.0f}")
print(f"  Median: ${df['RESALE_PRICE'].median():,.0f}")
print(f"  Min: ${df['RESALE_PRICE'].min():,.0f}")
print(f"  Max: ${df['RESALE_PRICE'].max():,.0f}")
print(f"  Missing: {df['RESALE_PRICE'].isnull().sum()}")



Dropped leakage columns: ['MIN_SELLING_PRICE', 'MEDIAN_RESALE_PRICE', 'MAX_SELLING_PRICE', 'RESALE_INDEX']


In [6]:
print("="*60)
print("DATA PREPROCESSING")
print("="*60)

# Step 1: Drop high-cardinality columns
print("\n1. Dropping high-cardinality columns...")
drop_cols = ['ADDRESS', 'POSTAL_CODE', 'NEAREST_MRT_NAME', 'SALE_QUARTER']
df = df.drop(columns=drop_cols, errors='ignore')
print(f"   Dropped {len(drop_cols)} columns")

# Step 2: Remove data leakage columns
print("\n2. Removing data leakage columns...")
leakage_cols = ['RESALE_INDEX', 'MIN_SELLING_PRICE', 'MEDIAN_RESALE_PRICE', 'MAX_SELLING_PRICE']
df = df.drop(columns=leakage_cols, errors='ignore')
print(f"   Removed {len(leakage_cols)} leakage columns")

# Step 3: Handle missing values
print("\n3. Handling missing values...")
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].median(), inplace=True)
print(f"   Filled missing values in {len([c for c in numeric_cols if c in df.columns])} numeric columns")

# Step 4: Encode categorical columns
print("\n4. Encoding categorical columns...")
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
if 'RESALE_PRICE' in categorical_cols:
    categorical_cols.remove('RESALE_PRICE')

print(f"   Found {len(categorical_cols)} categorical columns: {categorical_cols}")

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    print(f"     ✓ Encoded {col}")

# Verify preprocessing
print("\n5. Verification:")
print(f"   Final shape: {df.shape}")
print(f"   Object columns remaining: {df.select_dtypes(include=['object']).columns.tolist()}")
print(f"   Missing values: {df.isnull().sum().sum()}")

print("\n✅ Preprocessing complete!")


In [8]:
print("="*60)
print("TRAIN-TEST SPLIT")
print("="*60)

# Separate features and target
X = df.drop('RESALE_PRICE', axis=1)
y = df['RESALE_PRICE']

print(f"Features (X): {X.shape}")
print(f"Target (y): {y.shape}")

# Split data (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42
)

print(f"\nTraining set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

# Verify no object columns
object_check = X_train.select_dtypes(include=['object']).columns.tolist()
if len(object_check) > 0:
    print(f"\n⚠️ WARNING: Object columns found: {object_check}")
else:
    print("\n✅ All features are numeric - ready for training!")


In [10]:
print("="*60)
print("MODEL TRAINING")
print("="*60)

# Create XGBoost model
model = XGBRegressor(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1
)

print("Model parameters:")
print(f"  n_estimators: {model.n_estimators}")
print(f"  max_depth: {model.max_depth}")
print(f"  learning_rate: {model.learning_rate}")

# Train the model
print("\nTraining XGBoost model...")
model.fit(X_train, y_train)
print("✅ Model training complete!")


In [ ]:
print("="*60)
print("MODEL EVALUATION")
print("="*60)

# Make predictions
y_pred = model.predict(X_test)

# Calculate metrics
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100

print("\n📊 Performance Metrics:")
print(f"  RMSE: ${rmse:,.2f}")
print(f"  MAE: ${mae:,.2f}")
print(f"  R² Score: {r2:.4f}")
print(f"  MAPE: {mape:.2f}%")
print(f"  Model Accuracy: {r2*100:.2f}%")

# Calculate error statistics
errors = y_test - y_pred
print("\n📈 Error Analysis:")
print(f"  Mean Error: ${errors.mean():,.2f}")
print(f"  Std Dev of Errors: ${errors.std():,.2f}")
print(f"  Max Overestimation: ${errors.min():,.2f}")
print(f"  Max Underestimation: ${errors.max():,.2f}")


In [11]:
print("="*60)
print("FEATURE IMPORTANCE")
print("="*60)

# Get feature importance
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 20 Most Important Features:")
print(feature_importance.head(20).to_string(index=False))

# Plot feature importance
plt.figure(figsize=(12, 8))
top_features = feature_importance.head(20)
plt.barh(range(len(top_features)), top_features['importance'])
plt.yticks(range(len(top_features)), top_features['feature'])
plt.xlabel('Importance')
plt.title('Top 20 Most Important Features')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()



Training model...
Model training complete!


In [ ]:
print("="*60)
print("PREDICTION VISUALIZATION")
print("="*60)

# Create figure with multiple plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Actual vs Predicted
axes[0, 0].scatter(y_test, y_pred, alpha=0.5)
axes[0, 0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0, 0].set_xlabel('Actual Price')
axes[0, 0].set_ylabel('Predicted Price')
axes[0, 0].set_title('Actual vs Predicted Prices')
axes[0, 0].grid(True)

# Plot 2: Residuals
residuals = y_test - y_pred
axes[0, 1].scatter(y_pred, residuals, alpha=0.5)
axes[0, 1].axhline(y=0, color='r', linestyle='--')
axes[0, 1].set_xlabel('Predicted Price')
axes[0, 1].set_ylabel('Residuals')
axes[0, 1].set_title('Residual Plot')
axes[0, 1].grid(True)

# Plot 3: Error distribution
axes[1, 0].hist(residuals, bins=50, edgecolor='black')
axes[1, 0].set_xlabel('Prediction Error')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Distribution of Prediction Errors')
axes[1, 0].grid(True)

# Plot 4: Price distribution comparison
axes[1, 1].hist(y_test, bins=50, alpha=0.5, label='Actual', edgecolor='black')
axes[1, 1].hist(y_pred, bins=50, alpha=0.5, label='Predicted', edgecolor='black')
axes[1, 1].set_xlabel('Price')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Actual vs Predicted Price Distribution')
axes[1, 1].legend()
axes[1, 1].grid(True)

plt.tight_layout()
plt.show()

print("\n✅ Visualizations complete!")


In [ ]:
print("="*60)
print("SAMPLE PREDICTIONS")
print("="*60)

# Get sample predictions
sample_size = 10
sample_indices = np.random.choice(X_test.index, sample_size, replace=False)

results = pd.DataFrame({
    'Actual_Price': y_test.loc[sample_indices].values,
    'Predicted_Price': y_pred[y_test.index.get_indexer(sample_indices)],
})
results['Error'] = results['Actual_Price'] - results['Predicted_Price']
results['Error_%'] = (results['Error'] / results['Actual_Price']) * 100

print("\nRandom Sample Predictions:")
print(results.to_string(index=False))
